# Revenue Analysis of Cafe


In this notebook, we analyse the revenue of Cafe by analysing the monthly trend of revenue and customer behaviours. We also analyse the products that lead to customer churn. By calculating the standard deviation of intervals between two purchases of each customer plus, we define the churn threshold as the standard deviation plus the mean of intervals. After identifying the churned customers, we can find out the products that are likely to lead to customer churn.

The steps of analysis are as follows:
1. calculate the monthly trend of revenue
2. calculate the number of purchase and the total spending of each customers
3. calculate the retention rate and churn rate of customers
4. identify the most popular group of products in a single purchase
5. find out products that lead to customer churn

## load data and clean data 

In [ ]:
# read bigquery data into pandas dataframe
import pandas as pd
import pandas_gbq
import matplotlib.pyplot as plt
from collections import Counter
from itertools import combinations
#pd.set_option('display.max_colwidth', None)
df = pd.read_gbq(
    """
    SELECT  *
    FROM `jr-data-training.cafe.cafe-sales`
    --- LIMIT 10
  """,
    project_id="jr-data-training",
    location="australia-southeast1",
)
df.dropna(inplace=True)
df.drop(df[df['status'] != 2].index, inplace=True)

In [ ]:
item_date = pd.read_gbq(
    """
    SELECT  *
    FROM `jr-data-training.dbt_cafeanalytics.fact_product_sales`
    --- LIMIT 10
  """,
    project_id="jr-data-training",
    location="australia-southeast1",
)
item_date.dropna(inplace=True)

In [ ]:
item_details = pd.read_gbq(
    """
    SELECT  *
    FROM `jr-data-training.dbt_cafeanalytics.item_details`
    --- LIMIT 10
  """,
    project_id="jr-data-training",
    location="australia-southeast1",
)
item_details.dropna(inplace=True)

In [ ]:
customer = pd.read_gbq(
    """
    SELECT  *
    FROM `jr-data-training.dbt_cafeanalytics.customer`
    --- LIMIT 10
  """,
    project_id="jr-data-training",
    location="australia-southeast1",
)
customer.dropna(inplace=True)

In [ ]:
item_option = pd.read_gbq(
    """
    SELECT  *
    FROM `jr-data-training.dbt_cafeanalytics.item_options_analysis`
    --- LIMIT 10
  """,
    project_id="jr-data-training",
    location="australia-southeast1",
)
item_option.dropna(inplace=True)

In [ ]:
item_order = pd.read_gbq(
    """
    SELECT  *
    FROM `jr-data-training.dbt_cafeanalytics.item_options`
    --- LIMIT 10
  """,
    project_id="jr-data-training",
    location="australia-southeast1",
)
item_order.dropna(inplace=True)

## analyse revenue trend and customer behaviours

In [ ]:
# present monthly revenue trend
df['total'] = df['total'].astype(int)
df['date_paid'] = pd.to_datetime(df['date_paid'])

df['year_month'] = df['date_paid'].dt.to_period('M')
monthly_revenue = df.groupby('year_month')['total'].sum().reset_index()
monthly_revenue['year_month'] = monthly_revenue['year_month'].dt.to_timestamp()
monthly_revenue['total']=monthly_revenue['total'].astype(int)
# create graph of monthly revenue trend
plt.figure(figsize=(12, 6))
plt.plot(monthly_revenue['year_month'], monthly_revenue['total'], marker='o', linestyle='-')
plt.title('Monthly Revenue Trend')
plt.xlabel('Date')
plt.ylabel('Total Revenue')
plt.grid(True)
plt.gcf().autofmt_xdate()

plt.show()

In [ ]:
# calculate the number of customers and number of orders every month
monthly_customer_count = df.groupby('year_month')['customer_id'].nunique()
monthly_customer_count.index = monthly_customer_count.index.to_timestamp()
monthly_order_count = df.groupby('year_month')['order_id'].count()
monthly_order_count.index = monthly_order_count.index.to_timestamp()


In [ ]:
# create graph of the number of customers trend
plt.figure(figsize=(12,6))
plt.plot(monthly_customer_count.index, monthly_customer_count.values, marker='o', linestyle='-', color='b')
plt.title('number of customers every month')
plt.xlabel('date')
plt.ylabel('number of customers')
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()
# create graph of the number of order trend
plt.figure(figsize=(12,6))
plt.plot(monthly_order_count.index, monthly_order_count.values, marker='o', linestyle='-', color='b')
plt.title('number of orders every month')
plt.xlabel('date')
plt.ylabel('number of orders')
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

## analyse customer behaviours

In [ ]:
# calculate frequency distribution of total spending per customer
customer_total = df.groupby('customer_id')['total'].sum().reset_index()
total_stats = customer_total.describe()
customer_total.columns = ['customer_id', 'total_spent']
plt.figure(figsize=(10, 6))
plt.hist(customer_total['total_spent'], bins=50, color='skyblue', edgecolor='black')  
plt.xlabel('Total Spending')
plt.ylabel('Frequency')
plt.title('Frequency Distribution of Total Spending per Customer')
plt.grid(axis='y', alpha=0.75)  
plt.tight_layout()
plt.show()

In [ ]:
# calculate frequency distribution of the number of purchase
purchase_frequency = df.groupby('customer_id').size()
frequency_stats = purchase_frequency.describe()
sorted_customers = purchase_frequency.sort_values(ascending=False)
# create graph of frequency distribution of number of purchase
plt.figure(figsize=(10,6))
plt.hist(purchase_frequency, bins=40, edgecolor='black', alpha=0.7)
plt.title('number of purchase')
plt.xlabel('number of purchase')
plt.ylabel('number of customers')
plt.grid(True)
plt.show()

In [ ]:
#create a new column quarter
df['quarter'] = df['date_created'].dt.to_period('Q')

# number of order every quarter
quarterly_orders = df.groupby(['customer_id', 'quarter']).size().unstack(fill_value=0)

# at least one purchase every quarter
quarterly_active_customers = quarterly_orders.apply(lambda x: all(x > 0), axis=1)

retained_customers = quarterly_active_customers[quarterly_active_customers].index
print(retained_customers.tolist())

# calculate the the earliest date and the latest date
first_purchase = df.groupby('customer_id')['date_created'].min()
last_purchase = df.groupby('customer_id')['date_created'].max()

# calculate the lifecycle of customers
customer_lifecycle = (last_purchase - first_purchase).dt.days
# lifecycle is greater than 30 days
long_lived_customers = customer_lifecycle[customer_lifecycle > 30]
print('long lived customers:')
print(long_lived_customers)

## analyse the product performance

In [ ]:
# calculate the most popular item
item_option['pair_sum'] = item_option.groupby('item_name')['count'].transform('sum')
sorted_item = item_option.sort_values(by='pair_sum', ascending=False)
unique_sorted = sorted_item.drop_duplicates(subset='item_name')
print(unique_sorted[['item_name','pair_sum']])

In [ ]:
# present the most frequent items bought by customers sorted by total 
customer_item = pd.merge(df, customer, on = 'customer_id', how='inner')
customer_item.drop_duplicates(subset='customer_id',inplace=True)
print(customer_item[['customer_id','total','most_frequent_item']])

In [ ]:
# calculate the top 5 items
item_date['date_created'] = pd.to_datetime(item_date['date_created'])
item_date['year_month'] = item_date['date_created'].dt.to_period('M')
monthly_item = item_date.groupby(['year_month','item_name'])['quantity_sold'].sum().reset_index()
top_items = monthly_item.sort_values(['year_month','quantity_sold'], ascending=[True, False])
top_5_item = top_items.groupby('year_month').head(5)
print(top_5_item)

In [ ]:
# calculate the top 5 items in September
item_date['month'] = item_date['date_created'].dt.month
item_date['year'] = item_date['date_created'].dt.year
september_sales = item_date[item_date['month'] == 9 ]
top_sep_item = september_sales.groupby(['year','item_name'])['quantity_sold'].sum().reset_index()
top_sep_item = top_sep_item.sort_values(['year','quantity_sold'],ascending=[True, False])
top_5_sep_item = top_sep_item.groupby('year').head(5)
print(top_5_sep_item)

In [ ]:
# calculate the popular group of products in a single purchase
pairs = item_order.groupby('order_id')['item_name'].apply(list)
group_2 = Counter()
group_3 = Counter()
for items in pairs:
    unique_items = sorted(set(items))
    group_2.update(combinations(unique_items,2))
    group_3.update(combinations(unique_items,3))
# find the top 3 group
common_two = group_2.most_common(3)
common_three = group_3.most_common(3)
common_two_df = pd.DataFrame(common_two, columns=['item_pair', 'count'])
common_three_df = pd.DataFrame(common_three, columns=['item_pair', 'count'])


## calculate retention rate and churn rates

In [ ]:
#calculate the retention rate of customers
df['year'] = df['date_paid'].dt.to_period('Y')
# define the customers coming to the Cafe from 2019 and 2020 as the customer base used to calculate retention rate and churn rate
registered_customers = df[(df['year']=='2019') | (df['year']=='2020')]['customer_id'].unique()
retention_data = {}
# identify retained customers from 2021 to 2024
for year in range(2021,2024):
    year = str(year)
    active_customers = df[(df['year'] == year) & (df['date_paid'].notnull())]['customer_id'].unique()
    retained_customers = len(set(registered_customers) & set(active_customers))
    retention_data[year]=retained_customers
retained_customers_list = set(registered_customers) & set(active_customers) 
retained_customers_df = pd.DataFrame(retained_customers_list, columns=['customer_id'])

# identify churned customers from 2021 to 2024
churn_data = {}
for year in range(2021, 2024):
    year = str(year)
    churned_customers = len(set(registered_customers) - set(active_customers))
    churn_data[year] = churned_customers
churned_customers_list = set(registered_customers) - set(active_customers)
churned_customer_df = pd.DataFrame(churned_customers_list, columns=['customer_id'])
retention_rates = {}
churn_rates ={}
# calculate retention rate and churn rate
for year in range(2021, 2024):
    year = str(year)
    total_customers = len(registered_customers)
    retained_customers = retention_data[year]
    churned_customers = churn_data[year]
    retention_rate = retained_customers / total_customers if total_customers > 0 else 0
    churn_rate = churned_customers / total_customers if total_customers > 0 else 0
    retention_rates[year] = retention_rate
    churn_rates[year] = churn_rate
retention_df=pd.DataFrame({
    'Year': list(retention_rates.keys()),
    'Retention Rate': list(retention_rates.values()),
    'Churn Rate': list(churn_rates.values())
})
# create graph showing the retention rate trend and churn rate trend
plt.figure(figsize=(10, 6))

plt.plot(retention_df['Year'], retention_df['Retention Rate'], marker='o', label='Retention Rate', color='blue')
plt.plot(retention_df['Year'], retention_df['Churn Rate'], marker='o', label='Churn Rate', color='red')

plt.title('Customer Retention and Churn Rates (2021-2023)')
plt.xlabel('Year')
plt.ylabel('Rate')
plt.xticks(retention_df['Year'])  
plt.ylim(0, 1) 
plt.grid()
plt.legend()
plt.show()

In [ ]:
# find the products frequently bought by retained customers
filtered_orders = df[df['customer_id'].isin(retained_customers_df['customer_id'])]
filtered_order_ids = filtered_orders['order_id'].unique()
purchased_items = item_details[item_details['order_id'].isin(filtered_order_ids)]
item_counts = purchased_items['item_name'].value_counts()
top_10_retained_items = item_counts.head(10)
# find the products frequently bought by churned customers
filtered_orders_churn = df[df['customer_id'].isin(churned_customer_df['customer_id'])]
filtered_orders_churn_ids = filtered_orders_churn['order_id'].unique()
purchased_items_churn = item_details[item_details['order_id'].isin(filtered_orders_churn_ids)]
item_counts_churn = purchased_items_churn['item_name'].value_counts()
top_10_churned_items = item_counts_churn.head(10)
print(top_10_retained_items)

## analyse products that lead to customer churn

In [ ]:
df=df.sort_values(by=['customer_id','date_paid'])
# define interval as the days between two purchases
df['interval'] = df.groupby('customer_id')['date_paid'].diff().dt.days
intervals = df.groupby('customer_id')['interval'].apply(lambda x:x.dropna())
# define churn thresholds as the mean of intervals plus the standard deviation of intervals for each customer
churn_thresholds = intervals.groupby(level=0).agg(['mean','std']).reset_index()
churn_thresholds['churn_threshold'] = churn_thresholds['std'] * 2 + churn_thresholds['mean']
df['is_churned'] = False
# if interval is greater the threshold, then label the customer as churn
for customer_id, group in df.groupby('customer_id'):
    for i in range(1, len(group)):
        threshold_row = churn_thresholds[churn_thresholds['customer_id'] == customer_id]
        if not threshold_row.empty:
            churn_threshold = threshold_row['churn_threshold'].values[0]
            if group['interval'].iloc[i] > churn_threshold:
                df.loc[group.index[i-1], 'is_churned'] = True
# find the products that are likely to lead to customer churn
churn_orders = df[df['is_churned']].copy()
churned_order_ids = churn_orders['order_id'].unique()
churned_orders_details = item_details[item_details['order_id'].isin(churned_order_ids)]

product_churn_analysis = churned_orders_details['item_name'].value_counts().reset_index()
product_churn_analysis.columns=['item_name','churned_customer_count']
